# Environment Setup

In [1]:
!pip -q install einops timm torchmetrics  # already in most fastMRI envs

import os, math, time, csv, random, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, RandomSampler
from einops import rearrange
from skimage.metrics import structural_similarity as ssim   # ← NEW
from torch.amp import GradScaler # For the device-agnostic GradScaler
from torch.amp import autocast   # For the device-agnostic autocast

# Set random seeds for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


# Dataset Class Definition

In [2]:
class ProcessedFastMRIDataset(Dataset):
    """Dataset for loading preprocessed FastMRI data or creating from raw files"""
    
    def __init__(self, data_dir=None, file_list=None, mode='train', mask_func=None, use_processed=True):
        self.mode = mode
        self.use_processed = use_processed
        
        if use_processed:
            self.data_dir = os.path.join(data_dir, mode)
            try:
                all_files_in_dir = os.listdir(self.data_dir)
            except FileNotFoundError:
                raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

            # Initial list of all .pt files
            all_pt_files = sorted([os.path.join(self.data_dir, f) for f in all_files_in_dir if f.endswith('.pt')])

            # Filter out files with "metadata" in their name
            self.batch_files = []
            for f_path_str in all_pt_files:
                if "metadata" not in Path(f_path_str).name:
                    self.batch_files.append(f_path_str)
                else:
                    print(f"Filtering out metadata file: {f_path_str}")
            
            if not self.batch_files:
                raise FileNotFoundError(f"No valid .pt files (after filtering 'metadata' files) found in {self.data_dir}")
                
            self.examples = []
            print(f"Found {len(self.batch_files)} .pt files to process in {self.data_dir} after filtering.")
            
            for i, batch_file_path in enumerate(self.batch_files):
                print(f"Processing file: {batch_file_path}")
                try:
                    batch = torch.load(batch_file_path, map_location='cpu') # Load on CPU

                    if not isinstance(batch, dict):
                        print(f"  Warning: Skipped {batch_file_path}. Loaded object is not a dictionary (type: {type(batch)}).")
                        continue

                    if 'inputs' not in batch:
                        print(f"  Warning: Skipped {batch_file_path}. Missing 'inputs' key. Keys present: {list(batch.keys())}.")
                        continue
                    
                    if 'targets' not in batch: # It's good practice to check for targets too
                        print(f"  Warning: Skipped {batch_file_path}. Missing 'targets' key. Keys present: {list(batch.keys())}.")
                        continue
                        
                    num_samples = len(batch['inputs'])
                    self.examples.extend([(i, j) for j in range(num_samples)])
                    # print(f"  Successfully loaded {num_samples} samples from {batch_file_path}.")

                except Exception as e:
                    print(f"  Error loading or processing file {batch_file_path}: {e}. Skipping.")
            
            if not self.examples:
                raise ValueError(f"No valid examples could be loaded from {self.data_dir}. Check warnings above.")
            
            print(f"Successfully loaded a total of {len(self.examples)} examples.")

        else:
            # This part is for use_processed=False, ensure it's complete from your original code
            self.file_list = file_list
            self.mask_func = mask_func
            self.examples = []
            if not self.file_list:
                 raise ValueError("File list is empty when use_processed is False.")
            
            print(f"Processing {len(self.file_list)} raw files...")
            for fpath_obj in self.file_list: # Assuming file_list contains Path objects or strings
                fpath = str(fpath_obj) # Ensure it's a string for h5py
                try:
                    with h5py.File(fpath, 'r') as hf:
                        kspace = hf['kspace']
                        middle_slice = kspace.shape[0] // 2
                        self.examples.append((fpath, middle_slice))
                except Exception as e:
                    print(f"Error processing raw file {fpath}: {e}")
            
            if not self.examples:
                raise ValueError("No examples could be prepared from raw files.")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        if self.use_processed:
            batch_idx, sample_idx = self.examples[idx]
            batch_data = torch.load(self.batch_files[batch_idx], map_location='cpu')
            inputs = batch_data['inputs'][sample_idx]
            targets = batch_data['targets'][sample_idx]
        else:
            # This part is for use_processed=False, ensure it's complete from your original code
            fpath, slice_idx = self.examples[idx]
            
            with h5py.File(fpath, 'r') as hf:
                kspace = hf['kspace'][slice_idx]
                target_rss = hf['reconstruction_rss'][slice_idx] if 'reconstruction_rss' in hf else None
                
                kspace_tensor = T.to_tensor(kspace)
                # Fallback for max_value if not in attrs, ensure this value is appropriate
                max_value = hf.attrs.get('max', 0.00085) 
                
                if self.mask_func:
                    masked_kspace, mask, _ = T.apply_mask(kspace_tensor, self.mask_func)
                else:
                    masked_kspace = kspace_tensor
                    
                image = T.ifft2c(masked_kspace)
                image_abs = T.complex_abs(image)
                image_normalized = image_abs / max_value
                
                if target_rss is not None:
                    target_tensor = T.to_tensor(target_rss)
                    target_normalized = target_tensor / max_value
                else:
                    # Handle cases where target might be missing, perhaps raise error or return placeholder
                    # For now, let's assume target_rss is always present or this case is handled
                    raise ValueError(f"Target 'reconstruction_rss' not found in {fpath}")

                crop_shape = (320, 320) # Assuming T is fastmri.data.transforms
                inputs = T.center_crop(image_normalized, crop_shape)
                targets = T.center_crop(target_normalized, crop_shape)
        
        # Ensure inputs and targets are 2D tensors [height, width]
        # The model expects [batch_size, channels, height, width]
        # DataLoader will batch them. The model adds channel dim if input is 3D [B, H, W].
        # Here, __getitem__ should return single sample, typically [H,W] or [C,H,W]
        # If your model needs a channel dim here, .unsqueeze(0) might be needed for inputs/targets
        return inputs, targets


# Loss Functions and Metrics

In [3]:
class SSIMLoss(nn.Module):
    """SSIM loss module for MRI reconstruction"""
    def __init__(self, win_size=7, k1=0.01, k2=0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer('w', torch.ones(1, 1, win_size, win_size) / win_size**2)
        self.cov_norm = win_size**2 / (win_size**2 - 1)
    
    def forward(self, x, y):
        data_range = 1.0  # Images are normalized to [0,1]
        C1 = (self.k1 * data_range)**2
        C2 = (self.k2 * data_range)**2
        
        # Compute means
        ux = F.conv2d(x, self.w)
        uy = F.conv2d(y, self.w)
        
        # Compute variances and covariance
        uxx = F.conv2d(x * x, self.w)
        uyy = F.conv2d(y * y, self.w)
        uxy = F.conv2d(x * y, self.w)
        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)
        
        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2
        
        D = (A1 * A2) / (B1 * B2)
        return 1 - D.mean()

# Combined L1 and SSIM loss - common in MRI reconstruction
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()
        self.ssim_loss = SSIMLoss()
        
    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        ssim = self.ssim_loss(pred, target)
        return self.alpha * l1 + (1 - self.alpha) * ssim

# Calculate PSNR metric
def calculate_psnr(img1, img2):
    """Calculate PSNR between two images"""
    mse = torch.mean((img1 - img2) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

# Calculate SSIM metric for numpy images
def calculate_ssim(img1, img2):
    """Calculate SSIM between two numpy arrays"""
    return ssim(img1, img2, data_range=img1.max())


# Common Model Blocks

In [4]:
# %% [code]  Basic conv helpers used by every architecture
class DoubleConv(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU()
        )
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__(); self.pool=nn.AvgPool2d(2); self.conv=DoubleConv(in_ch,out_ch)
    def forward(self,x): return self.conv(self.pool(x))
import torch.nn.functional as F

class Up(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        self.conv=DoubleConv(in_ch,out_ch)
    # --- Up.forward (final, robust) ------------------------------------------
    def forward(self, x, skip):
        x = self.up(x)
    
        # 1) spatial size guard (already added earlier)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                              align_corners=False)
    
        # 2) channel-count guard – crop the *larger* tensor
        c = min(x.shape[1], skip.shape[1])     # desired channels after crop
        if x.shape[1] != c:
            x = x[:, :c, ...]                  # trim x if it is larger
        if skip.shape[1] != c:
            skip = skip[:, :c, ...]            # trim skip if it is larger
    
        return self.conv(torch.cat([x, skip], dim=1))
    



# Baseline U-Net

In [5]:
# %% [code]  Baseline UNet
# ────────────────────────────────────────────────────────────────────────────
class UNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, chans=32, num_pool_layers=4):
        super().__init__()

        self.inc = DoubleConv(in_chans, chans)            # 320
        self.downs = nn.ModuleList()
        ch = chans
        for _ in range(num_pool_layers - 1):              # 3 downs: 320→40
            self.downs.append(Down(ch, ch * 2))
            ch *= 2                                       # ch = 256 after loop

        self.bottleneck = DoubleConv(ch, ch * 2)          # 40→20, 256→512

        # ----- 1. decoder channel schedule 512→256→128→64 -----
        self.ups = nn.ModuleList([
            Up(ch * 2, ch),       # 512 → 256
            Up(ch,     ch // 2),  # 256 → 128
            Up(ch // 2, ch // 4)  # 128 →  64
        ])

        # ----- 2. output head: take 64-ch and map to `out_chans` -----
        self.outc = nn.Conv2d(ch // 4, out_chans, kernel_size=1)

    def forward(self, x):
        s0 = self.inc(x)                   # 320
        s1 = self.downs[0](s0)             # 160
        s2 = self.downs[1](s1)             #  80
        s3 = self.downs[2](s2)             #  40
        b  = self.bottleneck(s3)           #  20  (no skip recorded here!)

        u2 = self.ups[0](b,  s3)           #  40 ← 20
        u1 = self.ups[1](u2, s2)           #  80 ← 40
        u0 = self.ups[2](u1, s1)           # 160 ← 80
        out = self.outc(F.interpolate(u0, size=s0.shape[2:], mode='bilinear', align_corners=False))
        return out


# Transformer Primitives

In [6]:
# %% [code]  Transformer helpers
class CustomMLP(nn.Sequential):
    def __init__(self,dim,mlp_dim=None,p=0.):
        mlp_dim = mlp_dim or dim*4
        super().__init__(nn.Linear(dim,mlp_dim),nn.GELU(),nn.Dropout(p),
                         nn.Linear(mlp_dim,dim),nn.Dropout(p))
# In Cell 6
class TransformerBlock(nn.Module):
    def __init__(self,dim,heads=8,p=0.):
        super().__init__()
        self.n1=nn.LayerNorm(dim); self.attn=nn.MultiheadAttention(dim,heads,dropout=p,batch_first=True)
        # Changed MLP to CustomMLP here
        self.n2=nn.LayerNorm(dim); self.mlp=CustomMLP(dim,p=p)
    def forward(self,x):
        x=x+self.attn(self.n1(x),self.n1(x),self.n1(x))[0]
        return x+self.mlp(self.n2(x))


# Variant 1: BT-UNet

In [7]:
# %% BT-UNet
class BTUNet(nn.Module):
    def __init__(self, base_channels=32, depth=4, heads=8):
        super().__init__()
        self.core = UNet(chans=base_channels)
        # Explicit handles
        self.inc  = self.core.inc
        self.d1,self.d2,self.d3 = self.core.downs     # pooled encoder blocks
        self.bottleneck = self.core.bottleneck
        self.u2,self.u1,self.u0 = self.core.ups       # decoder blocks
        self.head = self.core.outc

        C = base_channels * 16                        # channels at bottleneck in/out
        self.tr = nn.Sequential(*[TransformerBlock(C, heads) for _ in range(depth)])

    def forward(self, x):
        s0 = self.inc(x)
        s1 = self.d1(s0)
        s2 = self.d2(s1)
        s3 = self.d3(s2)
        b  = self.bottleneck(s3)                      # B C 20 20
        B,C,H,W = b.shape
        b = rearrange(b, 'b c h w -> b (h w) c')
        b = self.tr(b)
        b = rearrange(b, 'b (h w) c -> b c h w', h=H, w=W)

        u2 = self.u2(b,  s3)
        u1 = self.u1(u2, s2)
        u0 = self.u0(u1, s1)
        out = self.head(F.interpolate(u0, size=s0.shape[2:], mode='bilinear', align_corners=False))
        return out

# Variant 2: UNETR

In [8]:
# %% UNETR
class PatchEmbed(nn.Module):
    def __init__(self,in_ch=1,patch=16,dim=768):
        super().__init__(); self.proj=nn.Conv2d(in_ch,dim,patch,patch)
    def forward(self,x):
        x=self.proj(x); B,C,H,W=x.shape
        return rearrange(x,'b c h w -> b (h w) c'),H,W
class UNETR(nn.Module):
    def __init__(self,im_size=320,patch=16,dim=768,depth=8,heads=12,dec_ch=(512,256,128,64)):
        super().__init__()
        self.patch_embed=PatchEmbed(1,patch,dim)
        self.pos=nn.Parameter(torch.zeros(1,(im_size//patch)**2,dim))
        self.blocks=nn.ModuleList([TransformerBlock(dim,heads) for _ in range(depth)])
        self.up4=nn.ConvTranspose2d(dim,dec_ch[0],2,2); self.c4=DoubleConv(dec_ch[0],dec_ch[0])
        self.up3=nn.ConvTranspose2d(dec_ch[0],dec_ch[1],2,2); self.c3=DoubleConv(dec_ch[1],dec_ch[1])
        self.up2=nn.ConvTranspose2d(dec_ch[1],dec_ch[2],2,2); self.c2=DoubleConv(dec_ch[2],dec_ch[2])
        self.up1=nn.ConvTranspose2d(dec_ch[2],dec_ch[3],2,2); self.c1=DoubleConv(dec_ch[3],dec_ch[3])
        self.out=nn.Conv2d(dec_ch[3],1,1)
    def forward(self,x):
        B,_,_,_=x.shape
        t,H,W=self.patch_embed(x); t=t+self.pos[:,:t.size(1)]
        for blk in self.blocks: t=blk(t)
        f=t.transpose(1,2).reshape(B,-1,H,W)
        x4=self.c4(self.up4(f)); x3=self.c3(self.up3(x4))
        x2=self.c2(self.up2(x3)); x1=self.c1(self.up1(x2))
        return self.out(x1)

# Variant 3: Swin-UNet

In [9]:
# %% Swin-UNet
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from timm.models.swin_transformer import WindowAttention, Mlp as MLP

def window_partition(x, window_size: int):
    """
    Partitions a tensor into non-overlapping windows.
    Args:
        x: Input tensor of shape (B, H, W, C).
        window_size (int): Size of the window.
    Returns:
        windows: Tensor of shape (num_windows*B, window_size, window_size, C).
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size: int, H: int, W: int):
    """
    Merges windows back into a tensor.
    Args:
        windows: Windows tensor of shape (num_windows*B, window_size, window_size, C).
        window_size (int): Size of the window.
        H (int): Original height of the feature map.
        W (int): Original width of the feature map.
    Returns:
        x: Tensor of shape (B, H, W, C).
    """
    # Calculate B assuming num_windows is (H/window_size) * (W/window_size)
    num_windows_h = H // window_size
    num_windows_w = W // window_size
    B = windows.shape[0] // (num_windows_h * num_windows_w)
    x = windows.view(B, num_windows_h, num_windows_w, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x
    
class SwinBlock(nn.Module): # Renamed to SwinBlock for direct replacement, or use SwinBlockFixed
    def __init__(self, dim, ws=8, heads=4): # ws is window_size
        super().__init__()
        self.dim = dim
        self.window_size = ws
        self.heads = heads

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim,
            num_heads=heads,
            window_size=(ws, ws),
            qkv_bias=True # A common default for Swin Transformers
        )
        self.norm2 = nn.LayerNorm(dim)
        # Assuming MLP is timm.models.layers.Mlp which takes in_features, hidden_features, etc.
        # self.mlp = MLP(dim) was in your code. If MLP is from timm, it should be:
        self.mlp = MLP(in_features=dim, hidden_features=int(dim * 4), act_layer=nn.GELU, drop=0.)


    def forward(self, x): # Expected input x: (B, C, H, W)
        B, C, H, W = x.shape
        if C != self.dim:
            raise ValueError(f"Input channel dimension {C} does not match block dimension {self.dim}")

        # Rearrange from (B, C, H, W) to (B, H, W, C) for window operations
        x_spatial = rearrange(x, 'b c h w -> b h w c')
        shortcut = x_spatial # Save shortcut in (B, H, W, C) format

        x_normed = self.norm1(x_spatial)

        # Pad feature maps to be divisible by window size
        H_pad, W_pad = H, W
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x_normed = F.pad(x_normed, (0, 0, pad_l, pad_r, pad_t, pad_b)) # Pads last dim (C), then W, then H
            H_pad += pad_b
            W_pad += pad_r
        
        # Partition windows
        x_windows = window_partition(x_normed, self.window_size)
        # x_windows: (num_windows*B, window_size, window_size, C)

        # Reshape for WindowAttention: (num_windows*B, N, C) where N = window_size*window_size
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # Window Multi-head Self-Attention (W-MSA)
        # Note: For Shifted Window MSA (SW-MSA), additional logic for cyclic shift and attention mask is needed.
        attn_windows = self.attn(x_windows, mask=None)
        # attn_windows: (num_windows*B, N, C)

        # Merge windows
        # Reshape back to (num_windows*B, window_size, window_size, C)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        merged_windows = window_reverse(attn_windows, self.window_size, H_pad, W_pad)
        # merged_windows: (B, H_pad, W_pad, C)

        # Remove padding if applied
        if pad_r > 0 or pad_b > 0:
            merged_windows = merged_windows[:, :H, :W, :].contiguous()
        
        # First residual connection (after attention)
        x_spatial = shortcut + merged_windows # Ensure shortcut has compatible shape

        # FFN part
        x_ffn = self.norm2(x_spatial)
        x_ffn = self.mlp(x_ffn)
        
        # Second residual connection (after FFN)
        x_out_spatial = x_spatial + x_ffn

        # Rearrange back to (B, C, H, W)
        x_out = rearrange(x_out_spatial, 'b h w c -> b c h w')
        
        return x_out

class SwinUNet(nn.Module):
    def __init__(self,base=32,ws=8):
        super().__init__()
        chs=[base,base*2,base*4,base*8]
        self.inc=nn.Conv2d(1,chs[0],3,padding=1)
        self.s1=self._stage(chs[0],chs[0],ws)
        self.d1=nn.Conv2d(chs[0],chs[1],2,2); self.s2=self._stage(chs[1],chs[1],ws)
        self.d2=nn.Conv2d(chs[1],chs[2],2,2); self.s3=self._stage(chs[2],chs[2],ws)
        self.d3=nn.Conv2d(chs[2],chs[3],2,2); self.s4=self._stage(chs[3],chs[3],ws)
        self.u3=nn.ConvTranspose2d(chs[3],chs[2],2,2); self.su3=self._stage(chs[2]*2,chs[2],ws)
        self.u2=nn.ConvTranspose2d(chs[2],chs[1],2,2); self.su2=self._stage(chs[1]*2,chs[1],ws)
        self.u1=nn.ConvTranspose2d(chs[1],chs[0],2,2); self.su1=self._stage(chs[0]*2,chs[0],ws)
        self.out=nn.Conv2d(chs[0],1,1)
    def _stage(self,in_ch,out_ch,ws):
        return nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False),
                             nn.InstanceNorm2d(out_ch,affine=True),nn.GELU(),
                             SwinBlock(out_ch,ws))
    def forward(self,x):
        x1=self.s1(self.inc(x))
        x2=self.s2(self.d1(x1))
        x3=self.s3(self.d2(x2))
        b =self.s4(self.d3(x3))
        u3=self.su3(torch.cat([self.u3(b),x3],1))
        u2=self.su2(torch.cat([self.u2(u3),x2],1))
        u1=self.su1(torch.cat([self.u1(u2),x1],1))
        return self.out(u1)

# Random Subset of Data

In [10]:
# %% [code]  helper – get DataLoader on a subset fraction
def make_loader(dataset,batch_size,fraction=1.0,shuffle=True):
    if fraction<1.0:
        n = int(len(dataset)*fraction)
        indices = random.sample(range(len(dataset)), n)
        subset = Subset(dataset, indices)
    else:
        subset = dataset
    return DataLoader(subset,batch_size=batch_size,shuffle=shuffle,
                      num_workers=4,pin_memory=True)


# Generic Trainer

In [11]:
# %% [code]  train_and_eval one model
def run_model(run_name, model_fn, train_ds, val_ds,
              batch_size=16, epochs=50, lr=1e-4, frac=0.25,
              out_root="./runs", device='cuda'):
    out_dir = Path(out_root)/run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    train_loader = make_loader(train_ds,batch_size,frac,shuffle=True)
    val_loader   = make_loader(val_ds,  batch_size,1.0, shuffle=False)

    model = model_fn().to(device)
    criterion = CombinedLoss(alpha=0.84).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',0.5,5)

    best = float('inf')
    log = []

    for ep in range(1,epochs+1):
        print(f"\n{run_name} | epoch {ep}/{epochs}")
        tr_loss = train_epoch(model,train_loader,optimizer,criterion,device)
        vl_loss, vl_psnr, vl_ssim = validate(model,val_loader,criterion,device)
        scheduler.step(vl_loss)
        log.append([ep,tr_loss,vl_loss,vl_psnr,vl_ssim])

        # save best
        if vl_loss < best:
            best = vl_loss
            torch.save({'epoch':ep,'state':model.state_dict()}, out_dir/'best.pth')

        # light checkpoint every 10
        if ep%10==0:
            torch.save({'epoch':ep,'state':model.state_dict()}, out_dir/f'ckpt_{ep}.pth')

    # save log
    with open(out_dir/'history.csv','w',newline='') as f:
        w=csv.writer(f); w.writerow(['epoch','train','val','psnr','ssim']); w.writerows(log)
    return out_dir


# Training and Validation Functions

In [12]:
def train_epoch(model, dataloader, optimizer, criterion, device, scaler, accumulation_steps=1): # Added scaler
    model.train()
    running_loss = 0.0
    # optimizer.zero_grad() # Moved inside the loop for gradient accumulation clarity if steps > 1

    with tqdm(dataloader, desc="Training") as pbar:
        for i, (inputs, targets) in enumerate(pbar):
            if i % accumulation_steps == 0: # Zero grad before starting a new accumulation cycle
                optimizer.zero_grad()

            inputs = inputs.to(device)
            targets = targets.to(device)

            if len(inputs.shape) == 3:
                inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3:
                targets = targets.unsqueeze(1)

            with autocast(device_type='cuda'): # Apply AMP to the forward pass
                outputs = model(inputs) # Error occurred here in the forward pass
                loss = criterion(outputs, targets)
                if accumulation_steps > 1:
                    loss = loss / accumulation_steps # Normalize loss

            scaler.scale(loss).backward() # Scale loss and backward pass

            if (i + 1) % accumulation_steps == 0: # Step optimizer after accumulation_steps
                scaler.step(optimizer)
                scaler.update()

            running_loss += loss.item() * (accumulation_steps if accumulation_steps > 1 else 1)
            pbar.set_postfix({'loss': loss.item() * (accumulation_steps if accumulation_steps > 1 else 1)})

    # Handle any remaining steps if dataloader size is not a multiple of accumulation_steps
    if len(dataloader) % accumulation_steps != 0:
        scaler.step(optimizer) # Perform the optimizer step with unscaled gradients
        scaler.update()
        # optimizer.zero_grad() # Already zeroed at the start of the next epoch or loop

    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    running_ssim = 0.0
    
    with torch.no_grad():
        with tqdm(dataloader, desc="Validation") as pbar:
            for inputs, targets in pbar:
                # Move tensors to the right device
                inputs = inputs.to(device)
                targets = targets.to(device)
                
                # Add channel dimension if needed
                if len(inputs.shape) == 3:
                    inputs = inputs.unsqueeze(1)
                if len(targets.shape) == 3:
                    targets = targets.unsqueeze(1)
                
                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                # Calculate metrics
                psnr = calculate_psnr(outputs, targets)
                
                # Update statistics
                running_loss += loss.item()
                running_psnr += psnr.item()
                
                # Calculate SSIM on CPU for one batch (it's slower)
                if running_ssim == 0:
                    for i in range(min(4, outputs.size(0))):  # Calculate for first 4 images only
                        output_np = outputs[i, 0].cpu().numpy()
                        target_np = targets[i, 0].cpu().numpy()
                        running_ssim += calculate_ssim(output_np, target_np)
                    running_ssim /= min(4, outputs.size(0))
                    
                pbar.set_postfix({'val_loss': loss.item(), 'psnr': psnr.item()})
                
    return running_loss / len(dataloader), running_psnr / len(dataloader), running_ssim


# Visualization Functions

In [13]:
def visualize_results(model, dataloader, device, epoch, output_dir):
    """Visualize model predictions on a few samples"""
    model.eval()
    
    # Get a batch of data
    inputs, targets = next(iter(dataloader))
    inputs = inputs.to(device)
    targets = targets.to(device)
    
    # Make predictions
    with torch.no_grad():
        outputs = model(inputs.unsqueeze(1) if len(inputs.shape) == 3 else inputs)
    
    # Create figure
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))
    fig.suptitle(f"FastMRI Reconstruction - Epoch {epoch}", fontsize=16)
    
    for i in range(4):  # Show 4 examples
        # Get images
        if i < inputs.size(0):
            input_img = inputs[i].cpu().numpy()
            output_img = outputs[i, 0].cpu().numpy()
            target_img = targets[i].cpu().numpy()
            
            # Display input
            axes[i, 0].imshow(input_img, cmap='gray')
            axes[i, 0].set_title(f"Input (Undersampled)")
            axes[i, 0].axis('off')
            
            # Display output
            axes[i, 1].imshow(output_img, cmap='gray')
            axes[i, 1].set_title(f"Prediction")
            axes[i, 1].axis('off')
            
            # Display target
            axes[i, 2].imshow(target_img, cmap='gray')
            axes[i, 2].set_title(f"Ground Truth")
            axes[i, 2].axis('off')
            
            # Calculate metrics
            psnr = calculate_psnr(
                torch.from_numpy(output_img), 
                torch.from_numpy(target_img)
            ).item()
            ssim_val = calculate_ssim(output_img, target_img)
            
            # Add metrics as text
            axes[i, 1].text(
                10, 20, 
                f'PSNR: {psnr:.2f} dB\nSSIM: {ssim_val:.4f}',
                color='white', fontsize=12, 
                bbox=dict(facecolor='black', alpha=0.5)
            )
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    
    # Save figure
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f"{output_dir}/epoch_{epoch}.png")
    plt.close()
    
    return fig


In [14]:
from pathlib import Path # Ensure Path is imported in this cell or globally

# ... (assuming _orig_init is correctly defined from the non-patched ProcessedFastMRIDataset.__init__)

def _patched_init(self, *args, **kwargs):
    _orig_init(self, *args, **kwargs)

    good_examples = []
    if hasattr(self, 'batch_files') and self.batch_files: # Check if batch_files exists and is not empty
        for idx, slice_idx in self.examples:
            # Ensure idx is a valid index for self.batch_files
            if 0 <= idx < len(self.batch_files):
                batch_file_string_path = self.batch_files[idx]
                # Convert the string path to a Path object
                path_obj = Path(batch_file_string_path)
                if "metadata" not in path_obj.name:  # Now path_obj.name is correct
                    good_examples.append((idx, slice_idx))
            else:
                print(f"Warning in patch: Invalid index {idx} for batch_files of length {len(self.batch_files)}")
        self.examples = good_examples
    elif not hasattr(self, 'batch_files') or not self.batch_files:
        # This case might occur if _orig_init failed to populate self.batch_files
        # or if use_processed was False (though the error trace suggests use_processed=True)
        print("Warning in patch: self.batch_files not populated or empty, skipping example filtering.")
        # self.examples would remain as whatever _orig_init set it to, or cause error if not set.

# ProcessedFastMRIDataset.__init__ = _patched_init # This line applies the patch
# print("Patched ProcessedFastMRIDataset to ignore files that contain 'metadata'")


# Main Training Script

In [15]:
# %% [code]  --- unified trainer that reuses *your* helpers & saves images ---
def run_model(run_name,
              model_fn,                        # lambda -> nn.Module
              train_ds, val_ds,
              batch_size=16, epochs=50, lr=1e-4,
              frac=1.0,                       # fraction of train set to sample
              viz_every=5,                    # call visualize_results every N epochs
              out_root="./runs",
              device='cuda'):

    out_dir = Path(out_root) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # --- dataloaders on a random subset for speed --------------------------------
    def make_loader(ds, shuffle, fraction=1.0):
        if fraction < 1.0:
            n = int(len(ds) * fraction)
            idx = random.sample(range(len(ds)), n)
            ds = Subset(ds, idx)
        return DataLoader(ds, batch_size=batch_size,
                          shuffle=shuffle, num_workers=4, pin_memory=True)

    train_loader = make_loader(train_ds, True,  frac)
    val_loader   = make_loader(val_ds,   False, 1.0)

    # --- model / optimiser / scheduler ------------------------------------------
    model     = model_fn().to(device)
    criterion = CombinedLoss(alpha=0.84).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', 0.5, 5)

    # --- bookkeeping ------------------------------------------------------------
    best_val = float('inf')
    hist = []                      # rows: [epoch, train, val, psnr, ssim]

    print(f"\n▶ {run_name} – training on {len(train_loader.dataset)} slices "
          f"({frac*100:.0f}% of full training set)")

    for ep in range(1, epochs + 1):
        print(f"\n{run_name} | epoch {ep}/{epochs}")

        tr_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_psnr, val_ssim = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        hist.append([ep, tr_loss, val_loss, val_psnr, val_ssim])

        # preview images
        if ep % viz_every == 0 or ep == epochs:
            visualize_results(model, val_loader, device,
                              epoch=ep, output_dir=out_dir)

        # save best checkpoint
        if val_loss < best_val:
            best_val = val_loss
            torch.save({'epoch': ep, 'state_dict': model.state_dict()},
                       out_dir / 'best.pth')

        # checkpoint every 10
        if ep % 10 == 0:
            torch.save({'epoch': ep, 'state_dict': model.state_dict()},
                       out_dir / f'ckpt_{ep}.pth')

    # --- save training curves ----------------------------------------------------
    hist = np.asarray(hist)           # shape = [E, 5]
    np.savetxt(out_dir / "history.csv",
               hist, delimiter=',',
               header="epoch,train,val,psnr,ssim", comments='')

    # plot & save figure
    plt.figure(figsize=(15, 4))
    plt.subplot(1, 3, 1); plt.plot(hist[:,0], hist[:,1], label='train'); plt.plot(hist[:,0], hist[:,2], label='val')
    plt.title("Loss"); plt.xlabel("epoch"); plt.legend()
    plt.subplot(1, 3, 2); plt.plot(hist[:,0], hist[:,3]); plt.title("PSNR (dB)"); plt.xlabel("epoch")
    plt.subplot(1, 3, 3); plt.plot(hist[:,0], hist[:,4]); plt.title("SSIM"); plt.xlabel("epoch")
    plt.tight_layout()
    plt.savefig(out_dir / "training_curves.png")
    plt.close()

    return out_dir


# Inference and Model Evaluation

In [16]:
def evaluate_model(model_path, dataloader, device, output_dir="./evaluation"):
    """Evaluate a trained model on a dataset"""
    # Load the model
    model = UNet(in_chans=1, out_chans=1, chans=32, num_pool_layers=4).to(device)
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize metrics
    psnrs = []
    ssims = []
    
    # Process batches
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(tqdm(dataloader, desc="Evaluating")):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Add channel dimension if needed
            if len(inputs.shape) == 3:
                inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3:
                targets = targets.unsqueeze(1)
            
            # Make predictions
            outputs = model(inputs)
            
            # Calculate metrics
            for j in range(outputs.size(0)):
                output_np = outputs[j, 0].cpu().numpy()
                target_np = targets[j, 0].cpu().numpy()
                
                psnr = calculate_psnr(outputs[j:j+1], targets[j:j+1]).item()
                ssim_val = calculate_ssim(output_np, target_np)
                
                psnrs.append(psnr)
                ssims.append(ssim_val)
            
            # Visualize first batch
            if i == 0:
                fig, axes = plt.subplots(4, 3, figsize=(15, 20))
                fig.suptitle("FastMRI Reconstruction Results", fontsize=16)
                
                for j in range(min(4, outputs.size(0))):
                    # Get images
                    input_img = inputs[j, 0].cpu().numpy()
                    output_img = outputs[j, 0].cpu().numpy()
                    target_img = targets[j, 0].cpu().numpy()
                    
                    # Display input
                    axes[j, 0].imshow(input_img, cmap='gray')
                    axes[j, 0].set_title(f"Input (Undersampled)")
                    axes[j, 0].axis('off')
                    
                    # Display output
                    axes[j, 1].imshow(output_img, cmap='gray')
                    axes[j, 1].set_title(f"Prediction")
                    axes[j, 1].axis('off')
                    
                    # Display target
                    axes[j, 2].imshow(target_img, cmap='gray')
                    axes[j, 2].set_title(f"Ground Truth")
                    axes[j, 2].axis('off')
                    
                    # Add metrics as text
                    axes[j, 1].text(
                        10, 20, 
                        f'PSNR: {psnrs[j]:.2f} dB\nSSIM: {ssims[j]:.4f}',
                        color='white', fontsize=12, 
                        bbox=dict(facecolor='black', alpha=0.5)
                    )
                
                plt.tight_layout()
                plt.subplots_adjust(top=0.95)
                plt.savefig(f"{output_dir}/evaluation_samples.png")
                plt.close()
    
    # Calculate average metrics
    avg_psnr = np.mean(psnrs)
    avg_ssim = np.mean(ssims)
    
    # Print results
    print(f"Average PSNR: {avg_psnr:.2f} dB")
    print(f"Average SSIM: {avg_ssim:.4f}")
    
    # Save metrics
    results = {
        'psnr': psnrs,
        'ssim': ssims,
        'avg_psnr': avg_psnr,
        'avg_ssim': avg_ssim
    }
    
    # Save in text file
    with open(f"{output_dir}/evaluation_results.txt", 'w') as f:
        f.write(f"U-Net Baseline Evaluation Results\n")
        f.write(f"Average PSNR: {avg_psnr:.2f} dB\n")
        f.write(f"Average SSIM: {avg_ssim:.4f}\n")
    
    # Plot histograms of metrics
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(psnrs, bins=20)
    plt.xlabel('PSNR (dB)')
    plt.ylabel('Count')
    plt.title(f'PSNR Histogram (Avg: {avg_psnr:.2f} dB)')
    
    plt.subplot(1, 2, 2)
    plt.hist(ssims, bins=20)
    plt.xlabel('SSIM')
    plt.ylabel('Count')
    plt.title(f'SSIM Histogram (Avg: {avg_ssim:.4f})')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/metric_histograms.png")
    plt.close()
    
    return results

# Example usage (uncomment to run)
# val_dataset = ProcessedFastMRIDataset(data_dir="./processed_fastmri_data", mode='val', use_processed=True)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
# results = evaluate_model("./unet_output/best_model.pth", val_loader, device)


# Run

In [17]:
# ─── data loaders (put in a cell ABOVE the pilot sweep) ────────────────────



In [18]:
import time, pandas as pd, numpy as np
from collections import OrderedDict
from torch.utils.data import DataLoader

if __name__ == "__main__":
    data_root = Path("/workspace/fastmri-reconstruction/processed_fastmri_data")
    train_ds  = ProcessedFastMRIDataset(data_root, mode='train', use_processed=True)
    val_ds    = ProcessedFastMRIDataset(data_root, mode='val',   use_processed=True)
    # In your main script/pilot sweep cell (e.g., cell [90] context)
    
    batch_size = 2                  # keep it the same as in run_model
    num_workers = 4                  # or whatever you normally use
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                              num_workers=0, pin_memory=True)
    
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=0, pin_memory=True)

    experiments = {
        'swin' : lambda: SwinUNet(base=16,ws=4),
        'unet' : lambda: UNet(chans=32),
        'bt'   : lambda: BTUNet(base_channels=32),
        'unetr': lambda: UNETR()
        
    }
    # ─── quick pilot sweep (≈10 epochs each) ───────────────────────────────────

    scaler = GradScaler(device='cuda')
    pilot_epochs = 10
    results = []
    
    for name, make_model in experiments.items():
        model = make_model().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
        criterion = torch.nn.L1Loss()
    
        t0 = time.time()
        for ep in range(1, pilot_epochs + 1):
            train_epoch(model, train_loader, optimizer, criterion, device, scaler, accumulation_steps=4) # Pass scaler and accumulation_steps
            val_loss, val_psnr, val_ssim = validate(model, val_loader,
                                                    criterion, device)
        sec_per_epoch = (time.time() - t0) / pilot_epochs
        results.append((name, val_loss, val_psnr, val_ssim, sec_per_epoch))
    
    # turn the raw list into a ranked DataFrame
    df = pd.DataFrame(results,
                      columns=["model", "loss", "psnr", "ssim", "sec/epoch"])
    
    # simple composite “score”: lower loss + higher psnr + higher ssim − speed penalty
    df["score"] = (
          (df["loss"].max() - df["loss"]) / (df["loss"].max() - df["loss"].min())
        + (df["psnr"] - df["psnr"].min())  / (df["psnr"].max() - df["psnr"].min())
        + (df["ssim"] - df["ssim"].min())  / (df["ssim"].max() - df["ssim"].min())
        - (df["sec/epoch"] - df["sec/epoch"].min())
          / (df["sec/epoch"].max() - df["sec/epoch"].min()) * 0.3   # speed weight 30 %
    )
    
    df = df.sort_values("score", ascending=False)
    display(df[["model", "score", "loss", "psnr", "ssim", "sec/epoch"]])


    # for name, make_model in experiments.items():
    #     run_model(name, make_model,
    #               train_ds, val_ds,
    #               batch_size=16, epochs=50, lr=1e-4,
    #               frac=0.25, viz_every=5, out_root="./runs", device=device)

    # # Evaluate the best checkpoints
    # val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4)
    # for name in experiments:
    #     evaluate_model(f'./runs/{name}/best.pth',
    #                    val_loader, device,
    #                    arch=name, output_dir=f'./runs/{name}/eval')


Filtering out metadata file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_metadata.pt
Found 61 .pt files to process in /workspace/fastmri-reconstruction/processed_fastmri_data/train after filtering.
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_batch_0000.pt
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_batch_0001.pt
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_batch_0002.pt
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_batch_0003.pt
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_batch_0004.pt
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_batch_0005.pt
Processing file: /workspace/fastmri-reconstruction/processed_fastmri_data/train/fastmri_train_4x_bat

Training:   0%|          | 0/61 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Training:   0%|          | 0/61 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

...

Training:   0%|          | 0/61 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Training:   0%|          | 0/61 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

,model,score,loss,psnr,ssim,sec/epoch
0,swin,2.787943,0.026889,28.720908,0.530560,38.535001
1,unet,2.530362,0.032329,26.843472,0.559588,25.443688
2,bt,2.059387,0.032817,26.298735,0.530993,53.814419
3,unetr,-0.091784,0.055743,22.041546,0.165329,34.123620
